# 第 12 章 · tf2：坐标变换树

本章回答：相机看到杯子在"相机前方 0.5 米"，机械臂要的是"杯子相对底座在哪"，这个换算谁来做？机器人身上几十个坐标系，怎么管？为什么 tf2 总报"变换在未来 / 过去"？

配套讲义：仓库里 `ros2/notes/lessons/12-tf2.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 tf2_ros。这里只用 math 写四元数和变换，算的是讲义第 1 步的两个数字例子；树查询和时间缓冲按讲义第 2 步的规则写成小函数。真代码里这些由 tf2 的 Buffer.lookup_transform 完成。

In [ ]:
import itertools
import math

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 单位四元数 | 不转是 (0, 0, 0, 1) | `IDENTITY` | (0, 0, 0, 1) |
| 第 1 步 只平移 | $P_{\text{base}}={}_{\text{base}}T_{\text{camera}}\cdot P_{\text{camera}}$ | `Transform((0.1, 0, 0.3)).apply((0.5, 0, 0))` | (0.6, 0, 0.3) |
| 第 1 步 加旋转 | 先旋转、后平移 | `Transform(t, quat_from_yaw(π/2)).apply(...)` | 先 (0, 0.5, 0)，再 (0.1, 0.5, 0.3) |
| 第 1 步 相乘 | ${}_{B}T_{A}={}_{B}T_{E}\cdot{}_{E}T_{A}$ | `B_T_E * E_T_A` | 一步到位 = 分两步 |
| 第 2 步 树查询 | 沿树找共同祖先，一路相乘 | `lookup(tree, 'map', 'camera_link')` | (5.0, 1.1, 0.3) |
| 第 2 步 时间缓冲 | 前后两条之间插值；超出报错 | `interpolate(buf, t)` | 1.5 → 0.5；2.1 future；0.5 past |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `Quat`**

In [ ]:
Quat = tuple[float, float, float, float]  # (x, y, z, w)，w 是实部

**常量 `IDENTITY`**

In [ ]:
IDENTITY: Quat = (0.0, 0.0, 0.0, 1.0)  # 不转

**常量 `Vec`**

In [ ]:
Vec = tuple[float, float, float]

**函数 `quat_conj`**：单位四元数的逆 = 共轭：虚部取负。

In [ ]:
def quat_conj(q: Quat) -> Quat:
    """单位四元数的逆 = 共轭：虚部取负。"""
    return (-q[0], -q[1], -q[2], q[3])

**函数 `quat_mul`**：四元数乘法：先转 b 再转 a。

In [ ]:
def quat_mul(a: Quat, b: Quat) -> Quat:
    """四元数乘法：先转 b 再转 a。"""
    ax, ay, az, aw = a
    bx, by, bz, bw = b
    return (aw * bx + ax * bw + ay * bz - az * by,
            aw * by - ax * bz + ay * bw + az * bx,
            aw * bz + ax * by - ay * bx + az * bw,
            aw * bw - ax * bx - ay * by - az * bz)

**函数 `rotate`**：用四元数转一个向量：q · (v, 0) · q⁻¹。

In [ ]:
def rotate(q: Quat, v: Vec) -> Vec:
    """用四元数转一个向量：q · (v, 0) · q⁻¹。"""
    x, y, z, _ = quat_mul(quat_mul(q, (*v, 0.0)), quat_conj(q))
    return (x, y, z)

**类 `Transform`**：父坐标系 ← 子坐标系：子坐标系的原点在父坐标系里的位置 t，加上朝向 q。

In [ ]:
class Transform:
    """父坐标系 ← 子坐标系：子坐标系的原点在父坐标系里的位置 t，加上朝向 q。"""

    def __init__(self, t: Vec = (0.0, 0.0, 0.0), q: Quat = IDENTITY):
        self.t, self.q = t, q

    def apply(self, p: Vec) -> Vec:
        """P_父 = T · P_子：先旋转，后平移。"""
        r = rotate(self.q, p)
        return (r[0] + self.t[0], r[1] + self.t[1], r[2] + self.t[2])

    def __mul__(self, other: "Transform") -> "Transform":
        """_B T_A = _B T_E · _E T_A：先用右边的，再用左边的。"""
        return Transform(self.apply(other.t), quat_mul(self.q, other.q))

    def inverse(self) -> "Transform":
        qi = quat_conj(self.q)
        t = rotate(qi, self.t)
        return Transform((-t[0], -t[1], -t[2]), qi)

**函数 `interpolate`**：按时间缓存的一个量（这里只缓存 x 平移）：在前后两条记录之间线性插值；超出范围就报错。

In [ ]:
def interpolate(buffer: list[tuple[float, float]], t: float) -> float:
    """按时间缓存的一个量（这里只缓存 x 平移）：在前后两条记录之间线性插值；超出范围就报错。"""
    times = [s for s, _ in buffer]
    if t > times[-1]:
        raise LookupError(f"extrapolation into the future：要 {t}，最新只有 {times[-1]}")
    if t < times[0]:
        raise LookupError(f"extrapolation into the past：要 {t}，最早只有 {times[0]}")
    for (t0, x0), (t1, x1) in itertools.pairwise(buffer):
        if t0 <= t <= t1:
            return x0 + (x1 - x0) * (t - t0) / (t1 - t0)
    return buffer[-1][1]

**函数 `lookup`**：tree[子] = (父, 父←子)。沿树往上找共同祖先，把一路的变换乘起来（需要时取逆）。返回 目标←源。

In [ ]:
def lookup(tree: dict[str, tuple[str, Transform]], target: str, source: str) -> Transform:
    """tree[子] = (父, 父←子)。沿树往上找共同祖先，把一路的变换乘起来（需要时取逆）。返回 目标←源。"""

    def to_root(frame: str) -> list[str]:
        chain = [frame]
        while chain[-1] in tree:
            chain.append(tree[chain[-1]][0])
        return chain

    up_s, up_t = to_root(source), to_root(target)
    common = next(f for f in up_s if f in up_t)
    root_from_source = Transform()
    for f in up_s[: up_s.index(common)]:  # 源 → 共同祖先
        root_from_source = tree[f][1] * root_from_source
    root_from_target = Transform()
    for f in up_t[: up_t.index(common)]:
        root_from_target = tree[f][1] * root_from_target
    return root_from_target.inverse() * root_from_source

**函数 `quat_from_yaw`**：只绕 z 轴转 yaw 弧度（平面机器人最常用）：(0, 0, sin(yaw/2), cos(yaw/2))。

In [ ]:
def quat_from_yaw(yaw: float) -> Quat:
    """只绕 z 轴转 yaw 弧度（平面机器人最常用）：(0, 0, sin(yaw/2), cos(yaw/2))。"""
    return (0.0, 0.0, math.sin(yaw / 2), math.cos(yaw / 2))

**函数 `r3`**：保留 3 位小数，去掉 -0.0，方便看。

In [ ]:
def r3(v) -> tuple:
    """保留 3 位小数，去掉 -0.0，方便看。"""
    return tuple(round(x, 3) + 0.0 for x in v)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 不转的四元数是 (0, 0, 0, 1)；绕 z 转 90° 是 (0, 0, sin45°, cos45°)")
print("  ", IDENTITY, "；", r3(quat_from_yaw(math.pi / 2)))

In [ ]:
print("\n第一步（续）· 相机在底座前 0.1 m、上 0.3 m，没转；杯子在相机前方 0.5 m")
base_T_camera = Transform((0.1, 0.0, 0.3))
print("  P_base =", r3(base_T_camera.apply((0.5, 0.0, 0.0))))

In [ ]:
print("\n第一步（再续）· 同一个相机朝左转 90°：先旋转、后平移")
base_T_camera_yaw = Transform((0.1, 0.0, 0.3), quat_from_yaw(math.pi / 2))
print("  先转:", r3(rotate(base_T_camera_yaw.q, (0.5, 0.0, 0.0))), "→ 再加平移:",
      r3(base_T_camera_yaw.apply((0.5, 0.0, 0.0))))

In [ ]:
print("\n第一步（又续）· 变换相乘：_B T_A = _B T_E · _E T_A")
E_T_A = Transform((1.0, 0.0, 0.0), quat_from_yaw(math.pi / 2))
B_T_E = Transform((0.0, 2.0, 0.0))  # 两个变换是编的
B_T_A = B_T_E * E_T_A
p = (1.0, 0.0, 0.0)
print("  一步到位:", r3(B_T_A.apply(p)), "；分两步:", r3(B_T_E.apply(E_T_A.apply(p))))

In [ ]:
print("\n第二步 · REP 105 的树：map → odom → base_link → camera_link，查 map ← camera_link")
tree = {
    "odom": ("map", Transform((2.0, 1.0, 0.0))),  # 定位修正（编的）
    "base_link": ("odom", Transform((3.0, 0.0, 0.0), quat_from_yaw(math.pi / 2))),  # 里程计（编的）
    "camera_link": ("base_link", Transform((0.1, 0.0, 0.3))),  # 安装位置：第一步的数
}
map_T_cam = lookup(tree, "map", "camera_link")
print("  相机原点在 map 里:", r3(map_T_cam.t))
print("  反过来查 camera_link ← map，再乘回去是不是单位变换:",
      r3((lookup(tree, "camera_link", "map") * map_T_cam).t))

In [ ]:
print("\n第二步（续）· 时间缓冲：x 在 t=1.0 是 0.0，在 t=2.0 是 1.0")
buf = [(1.0, 0.0), (2.0, 1.0)]
print("  t=1.5 →", interpolate(buf, 1.5))
for t in [2.1, 0.5]:
    try:
        interpolate(buf, t)
    except LookupError as e:
        print(f"  t={t} →", e)

## 3. 代码里才看得到的

1. **四元数用的是半角**：转 90° 时 $z=\sin 45°\approx 0.707$，不是 $\sin 90°=1$。四个数平方和永远是 1。
2. **`Transform` 存的是"父 ← 子"**：`tree['base_link'] = ('odom', T)` 里的 T 把 base_link 里的点换到 odom。`lookup(目标, 源)` 的参数顺序和 tf2 的 `lookup_transform` 一样：目标在前。
3. **树里 map → odom、odom → base_link 的数是编的**；base_link → camera_link 用的是第 1 步的安装位置。
4. **时间缓冲只缓存了一个数（x 平移）**：真 tf2 对平移做线性插值、对旋转做球面插值（slerp）。

## 4. 自己改一改

- 把相机改成朝右转 90°（`quat_from_yaw(-math.pi / 2)`），看杯子在底座坐标系里到了哪；
- 在 `tree` 里再挂一个 `gripper`（父是 `base_link`），用 `lookup` 查 `camera_link ← gripper`，看共同祖先是谁。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **树**：每个坐标系一个爹，不能有环；查询沿树走，乘起来。
- **变换**：先转后移；四元数存，欧拉角看；手算完要归一化。
- **两种 tf**：会动的发 `/tf`，不动的发 `/tf_static`。
- **REP 105**：map 到 odom 管准（会跳），odom 到 base_link 管顺（会漂），base_link 往下来自 URDF。
- **查询**：目标在前、源在后；时间给 0 拿最新；报未来过去先查时间戳和 sim time。